# Tugas 1A: Advanced TF-IDF & Text Summarization (Danantara)

**Objective:** Memahami konsep TF-IDF secara mendalam, analisis kata spesifik 'investasi', dan implementasi Text Summarization sederhana.

## 1. Persiapan Data & Library

In [ ]:
import pandas as pd
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
from sklearn.metrics.pairwise import cosine_similarity

# Korpus Teks Danantara
corpus = [
    "Presiden Prabowo Subianto resmi meluncurkan Badan Pengelola Investasi Daya Anagata Nusantara atau Danantara",
    "Danantara akan mengelola aset negara dan menjadi pilar utama ekonomi Indonesia",
    "Target dana kelolaan Danantara direncanakan mencapai sembilan ratus miliar dolar",
    "Prabowo berharap Danantara dapat meningkatkan kemakmuran dan daya saing bangsa melalui investasi"
]

# Stopwords Bahasa Indonesia sederhana
stopwords = set(["dan", "atau", "untuk", "di", "ke", "dari", "ini", "itu", "adalah", "akan", "dapat", "pada"])

print("Corpus Danantara loaded.")

## 2. Preprocessing & Tokenization

In [ ]:
def preprocess(text, remove_stop=True):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = text.split()
    if remove_stop:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens

tokenized_corpus = [preprocess(d) for d in corpus]
vocab = sorted(list(set([word for doc in tokenized_corpus for word in doc])))

print(f"Vocabulary size: {len(vocab)}")

## 3. Perhitungan Manual TF-IDF

In [ ]:
def get_tf_matrix(token_docs, vocabulary):
    tf_matrix = []
    for doc in token_docs:
        row = []
        total_terms = len(doc)
        for word in vocabulary:
            count = doc.count(word)
            row.append(count / total_terms if total_terms > 0 else 0)
        tf_matrix.append(row)
    return np.array(tf_matrix)

def get_idf_vector(token_docs, vocabulary):
    N = len(token_docs)
    idf_vector = []
    for word in vocabulary:
        df = sum(1 for doc in token_docs if word in doc)
        # Gunakan log10 untuk perhitungan manual standard
        idf_vector.append(math.log10(N / df))
    return np.array(idf_vector)

tf_matrix = get_tf_matrix(tokenized_corpus, vocab)
idf_vector = get_idf_vector(tokenized_corpus, vocab)
tfidf_matrix = tf_matrix * idf_vector

df_tfidf = pd.DataFrame(tfidf_matrix, columns=vocab, index=[f"Doc {i+1}" for i in range(len(corpus))])

## 4. Analisis Kata Spesifik: 'investasi'
Sesuai instruksi, kita akan melihat bobot TF-IDF untuk kata **'investasi'** di setiap dokumen.

In [ ]:
target_word = 'investasi'
if target_word in df_tfidf.columns:
    word_scores = df_tfidf[target_word]
    print(f"Skor TF-IDF untuk kata '{target_word}':")
    print(word_scores)
    
    plt.figure(figsize=(8, 4))
    sns.barplot(x=word_scores.index, y=word_scores.values, palette="magma")
    plt.title(f"TF-IDF Score of '{target_word}' across Documents")
    plt.ylabel("TF-IDF Score")
    plt.show()
else:
    print(f"Kata '{target_word}' tidak ditemukan dalam vocab (mungkin terhapus stopword).")

## 5. Implementasi Text Summarization (Sederhana)
Summarization dilakukan dengan merata-ratakan skor TF-IDF kata-kata yang ada dalam setiap kalimat. Kalimat dengan skor tertinggi dianggap sebagai ringkasan (inti) dari teks.

In [ ]:
sentence_scores = []
for i in range(len(corpus)):
    # Hitung rata-rata skor TF-IDF non-nol dalam dokumen
    scores = tfidf_matrix[i]
    avg_score = np.mean(scores[scores > 0]) if any(scores > 0) else 0
    sentence_scores.append(avg_score)

df_summary = pd.DataFrame({
    'Sentence': corpus,
    'Importance Score': sentence_scores
}).sort_values(by='Importance Score', ascending=False)

print("Hasil Summarization (Kalimat paling penting ke kurang penting):")
display(df_summary)

best_sentence = df_summary.iloc[0]['Sentence']
print(f"\nRingkasan Utama: {best_sentence}")

## 6. Visualisasi Heatmap TF-IDF

In [ ]:
plt.figure(figsize=(15, 6))
sns.heatmap(df_tfidf, annot=True, cmap="YlGnBu")
plt.title("Full TF-IDF Heatmap (Danantara)")
plt.show()

## 7. Kesimpulan
1. **Kata 'investasi'**: Memiliki skor yang bervariasi. Jika muncul di dokumen yang lebih pendek, skor TF-IDF-nya akan lebih tinggi (karena TF lebih besar).
2. **Summarization**: Kalimat yang mengandung banyak kata-kata 'unik' (skor IDF tinggi) akan mendapatkan skor kepentingan yang lebih tinggi.
3. **Regex & Preprocessing**: Tetap digunakan untuk memastikan pembersihan data yang optimal.